In [44]:
import subprocess
subprocess.run(['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'])

CompletedProcess(args=['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'], returncode=0)

In [45]:
import numpy as np
from collections import Counter
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)
alloy = read('bulk_structure/MgZn2_relaxed.cif')
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 12 atoms, Mg4Zn8


In [46]:
slab = surface(alloy, (1, 0, 1), 6, vacuum=8)
sc = make_supercell(slab, [[2,0,0],[0,2,0],[0,0,1]])

sym = np.array(sc.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')

ps = AseAtomsAdaptor().get_structure(sc)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(sc)}, Mg: {mg}, Zn: {zn}")
print(f"Stoichiometric: {'YES' if zn == 2*mg else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 288, Mg: 96, Zn: 192
Stoichiometric: YES
Symmetric: False


In [53]:
z = sc.get_positions()[:, 2] #gets z coordinate of every atom in slab. get_positions([:, 2] takes just the z column from array
sym = np.array(sc.get_chemical_symbols()) #gets whether its mg or zn
order = np.argsort(z) #gives atom indices sorted by z coordinate so its in order. order[0] is the index of the atom closest to the bottom, order[-1] is the topmost.

layers = []
layer_idx = []
cur = [order[0]] #starts new layer group with the bottomost atom 
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < 0.3: #loop goes through atoms in z order, if atom is within 0.3A of last atom they are in same layer
        cur.append(order[i]) #when we go to a new layer we save the layers composition and indices and move to another cur
    else:
        layers.append(Counter(sym[j] for j in cur)) #just like atom counter- Mg 4 Zn 4 for example
        layer_idx.append(list(cur)) #actually holds atom indices in each plane
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(10, n//2)): #comparsion loop- compares layer 0 to layer 47 (top layer) and does this for first 10 pairs. to see if they align
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]]) #average z position for atoms in that layer
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    comp_b = f"Mg{bot.get('Mg',0)}Zn{bot.get('Zn',0)}" if bot.get('Mg',0) and bot.get('Zn',0) else (f"Mg{bot.get('Mg',0)}" if bot.get('Mg',0) else f"Zn{bot.get('Zn',0)}")
    comp_t = f"Mg{top.get('Mg',0)}Zn{top.get('Zn',0)}" if top.get('Mg',0) and top.get('Zn',0) else (f"Mg{top.get('Mg',0)}" if top.get('Mg',0) else f"Zn{top.get('Zn',0)}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")
    #gives distances from top layer and distances from bottom layer
    #for symmetric, a layer at distance 1.3A from bottom surface and a layer at 1.3A from top surface should have same composition and match

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0        Zn8     0.000  |       47        Zn4     0.000 NO <---
      1        Mg4     0.606  |       46        Zn8     0.330 NO <---
      2     Mg4Zn4     1.318  |       45        Mg4     0.936 NO <---
      3        Zn4     1.709  |       44     Mg4Zn4     1.648 NO <---
      4     Mg4Zn4     2.100  |       43        Zn4     2.039 NO <---
      5        Mg4     2.812  |       42     Mg4Zn4     2.430 NO <---
      6        Zn8     3.417  |       41        Mg4     3.142 NO <---
      7        Zn4     3.748  |       40        Zn8     3.748 NO <---
      8        Zn8     4.078  |       39        Zn4     4.078 NO <---
      9        Mg4     4.684  |       38        Zn8     4.409 NO <---


In [54]:
# The top surface has an extra Zn4 layer that the bottom doesn't.
# Removing it gives a symmetric (but non-stoichiometric) slab.
# From that symmetric slab we extract the inversion operation,
# which tells us exactly where atoms must go to maintain symmetry.

top_zn4_mask = z > 31.9 #creates a T/F array for every atom,top Zn4 is at 32.1A so anything above 31.9 is in this layer
keep = np.where(~top_zn4_mask)[0] #flips the true to false so we keep all atoms that are not in Zn4 layer
trimmed = sc[keep] #removed the zn4 and made new ase atoms object 

ps_trim = AseAtomsAdaptor().get_structure(trimmed) #converts from ase format to pymatgen so we can check symmetry
slab_trim = Slab(ps_trim.lattice, ps_trim.species, ps_trim.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_trim, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"After removing top Zn4:")
print(f"Atoms: {len(trimmed)}, Symmetric: {slab_trim.is_symmetric()}")

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1) #Analyses the trimmed slab and finds all its symmetry operations, 0.1 is tolerance in A for matching positions
print(f"Space group: {sga.get_space_group_symbol()}")

# Find the inversion operation (rotation = -I)
for op in sga.get_symmetry_operations(): #returns a list of symmetry operations the slab has, rotation matrix and translational vector. describes transformation in fractional coordinates
    if op.rotation_matrix[2][2] < 0: #looking for inversion operation, has rotation matrix -I. Inversion maps every atom at position f to t-f where t is translational vector
        inv_translation = op.translation_vector #This is the t in the formula above. The inversion centre sits at t/2 in fractional coordinates
        print(f"\nInversion operation found:")
        print(f"  Maps fractional coord f -> {inv_translation} - f")
        break

After removing top Zn4:
Atoms: 284, Symmetric: True
Space group: C2/m

Inversion operation found:
  Maps fractional coord f -> [0.66421053 0.16526946 0.99176955] - f


In [55]:
# The 4 top Zn atoms. We keep 2 on top, move 2 to the inversion
# partner positions of the kept ones. This restores stoichiometry
# while preserving the inversion symmetry.

ps_sc = AseAtomsAdaptor().get_structure(sc) #converts ase atoms object to pymatgen as it uses fractional corrdinates which the inversion does
top_zn_ids = [209, 65, 281, 137] #The atom indices of the 4 Zn in the top Zn4 layer, identified from cell 3.

# Calculate inversion partner positions for each
inv_positions_cart = {}
print("Top Zn4 and their inversion partners:\n")
for idx in top_zn_ids:
    frac = ps_sc[idx].frac_coords #Gets the fractional coordinates (values between 0 and 1 relative to the cell vectors) of atom idx
    inv_frac = inv_translation - frac #Applies the inversion. The operation is f → T - f, where T = [0.664, 0.165, 0.992] from cell 4.
    inv_frac = inv_frac % 1.0 #Wraps back into the unit cell if any coordinate went negative or above 1.
    inv_cart = ps_sc.lattice.get_cartesian_coords(inv_frac) #Converts back to Cartesian xyz coordinates in A so we can set the ASE atom positions
    inv_positions_cart[idx] = inv_cart #stores result
    cart = ps_sc.lattice.get_cartesian_coords(frac)
    print(f"  idx={idx}: ({cart[0]:.3f}, {cart[1]:.3f}, {cart[2]:.3f}) "
          f"-> inv: ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

# Keep 209, 281 on top surface
# Move 65 -> inversion partner of 209
# Move 137 -> inversion partner of 281
sc_fixed = sc.copy()
sc_fixed.positions[65] = inv_positions_cart[209] 
sc_fixed.positions[137] = inv_positions_cart[281]

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg = sum(sym_f == 'Mg')
zn = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAfter reconstruction:")
print(f"Atoms: {len(sc_fixed)}, Mg: {mg}, Zn: {zn}")
print(f"Stoichiometric: {'YES' if zn == 2*mg else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")

Top Zn4 and their inversion partners:

  idx=209: (13.982, 1.719, 32.138) -> inv: (2.805, 10.280, 7.670)
  idx=65: (3.797, 1.719, 32.138) -> inv: (12.991, 10.280, 7.670)
  idx=281: (15.379, 6.868, 32.138) -> inv: (1.408, 5.131, 7.670)
  idx=137: (5.194, 6.868, 32.138) -> inv: (11.593, 5.131, 7.670)

After reconstruction:
Atoms: 288, Mg: 96, Zn: 192
Stoichiometric: YES
Symmetric: True


In [52]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [56]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min() #Gets the z-coordinate of every atom, finds the highest and lowest, and subtracts to give the slab thickness in A
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1])) #area calculation

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mgzn2_101_6layers_reconstructed_ase.data")

print(f"6-layer (10-11) slab saved")
print(f"  File: slabs/mgzn2_101_6layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

6-layer (10-11) slab saved
  File: slabs/mgzn2_101_6layers_reconstructed_ase.data
  Atoms: 288
  Thickness: 24.5 A
  Area: 209.8 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [58]:
#unrelaxed surface energy calculation
E_slab = -370.77774         # eV
E_bulk_per_fu = -16.195843 / 4   # eV per formula unit = -4.048961 eV
n = 288 / 3                 # formula units in slab = 32
A = 209.8              # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -4.048961 eV
n * E_bulk = -388.70023 eV
E_slab - n*E_bulk = 17.92249 eV
S = 0.042713 eV/Ang^2
S = 0.6843 J/m^2


In [59]:
#relaxed surface energy calculation
E_slab_relaxed =  -371.600815905945  # eV
E_bulk_per_fu = -16.195843 / 4  # eV per formula unit
n = 288 / 3                      # 32 formula units
A = 209.8                 # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.6843
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 17.09942 eV
S relaxed = 0.040752 eV/Ang^2
S relaxed = 0.6529 J/m^2

Unrelaxed S = 0.6843 J/m^2
Relaxed S   = 0.6529 J/m^2
Relaxation energy = 0.0314 J/m^2
